In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Read data

In [2]:
with open('../input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print(f'Characters in text: {len(text)}')

Characters in text: 1115394


In [4]:
print(f'{text[:100]}')

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


# Getting unique characters and vocabulary size

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


# Character level tokenizer

In [6]:
encoded_dict = {chars[i]: i for i in range(vocab_size)}
decoded_dict = {i: chars[i] for i in range(vocab_size)}

encode = lambda s: [encoded_dict[c] for c in s]
decode = lambda l: "".join([decoded_dict[i] for i in l])

print(encode("di si kompa"))
print(decode(encode("di si kompa")))

[42, 47, 1, 57, 47, 1, 49, 53, 51, 54, 39]
di si kompa


In [7]:
data = encode(text)
data = np.array(data, dtype=np.int64)
print(data.shape)

(1115394,)


# Splitting train and val

In [8]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size+1]

array([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'when input is {context}, target is {target}')

when input is [18], target is 47
when input is [18 47], target is 56
when input is [18 47 56], target is 57
when input is [18 47 56 57], target is 58
when input is [18 47 56 57 58], target is 1
when input is [18 47 56 57 58  1], target is 15
when input is [18 47 56 57 58  1 15], target is 47
when input is [18 47 56 57 58  1 15 47], target is 58


In [11]:
np.random.seed(1337)

batch_size = 4
block_size = 8

def get_batch(split: str):
    data = train_data if split == "train" else val_data
    idx = np.random.randint(len(data) - block_size, size=batch_size)
    x = np.vstack([data[i:i+block_size] for i in idx])
    y = np.vstack([data[i+1:i+block_size+1] for i in idx])
    return x, y

xb, yb = get_batch("train")
print(xb.shape)
print(xb, "\n----------------------")
print(yb.shape)
print(yb, "\n----------------------")

(4, 8) (4, 8)
